# GraphRAG

## Import packages

In [1]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import json
from dotenv import load_dotenv
from getpass import getpass

from neurorag.neurorag import NeuroRAG

from metrics import (
  embeddings_cosine_sim_metric,
  bleu_metric,
  rogue_l_metric,
  rogue_1_metric,
  factscore_metric,
  summac_zs_metric,
  summac_conv_metric,
)

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
2026-01-14 01:00:28,874 - INFO - Using default tokenizer.
2026-01-14 01:00:28,874 - INFO - Using default tokenizer.


## Disable warnings

In [2]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook:
1. MISTRAL_API_KEY
2. OPENAI_API_KEY
3. OPENAI_PROXY
4. TAVILY_API_KEY
5. ENTREZ_EMAIL

## Import packages

In [3]:
env_variables = [
  'MISTRAL_API_KEY',
  'OPENAI_API_KEY',
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Build model

In [4]:
app = NeuroRAG(debug=False)
app.compile()

2026-01-14 01:00:29,080 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-01-14 01:00:30,880 - INFO - HTTP Request: GET https://api.trychroma.com/api/v2/auth/identity "HTTP/1.1 200 OK"
2026-01-14 01:00:30,881 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-01-14 01:00:32,387 - INFO - HTTP Request: GET https://api.trychroma.com/api/v2/tenants/3355f833-8e5a-4281-a23f-d603cf0bfad1 "HTTP/1.1 200 OK"
2026-01-14 01:00:32,608 - INFO - HTTP Request: GET https://api.trychroma.com/api/v2/tenants/3355f833-8e5a-4281-a23f-d603cf0bfad1/databases/neurorag "HTTP/1.1 200 OK"
2026-01-14 01:00:32,835 - INFO - HTTP Request: POST https://api.trychroma.com/api/v2/tenants/3355f833-8e5a-4281-a23f-d603cf0bfad1/databases/neurorag/collections "HTTP/1.1 200 OK"


## Evaluate RAG

### Load QA dataset

In [5]:
mediqa_df = pd.read_csv('../datasets/neurobiology_mediqa.csv')
mediqa_df

,question,answer
0,SSPE. My son is 33years of age and did not hav...,Subacute sclerosing panencephalitis: Subacute ...
1,Homozygout MTHFR A1298C Health Issues and long...,MTHFR gene variant (Inheritance): Because each...
2,What is Stroke?,Stroke: A stroke occurs when the blood supply ...
3,What causes Stroke?,Ischemic Stroke (Summary): Summary A stroke is...
4,What are the symptoms of Stroke?,What are the symptoms of Stroke?: The signs an...
5,What are the treatments of Stroke?,Stroke (Treatment): A stroke is a medical emer...
6,What is Dementia?,Dementia (WHAT IS DEMENTIA?): Dementia is the ...
7,What causes Dementia?,What causes Dementia?: Dementia usually occurs...
8,What are the symptoms of Dementia?,Dementia (Symptoms): Dementia symptoms include...
9,How to diagnose Dementia?,Dementia (Diagnosis): Diagnosing dementia and ...


### Load cached RAGs responses

In [6]:
cache_path = Path('cache.json')

if not os.path.exists(cache_path):
  data = {}
  with open(cache_path, 'w') as file:
    json.dump(data, file)

with open(cache_path, 'r') as f:
  cache = json.load(f)

CACHE_KEY = 'text-to-text-neurorag-evaluation'

if CACHE_KEY not in cache:
  cache[CACHE_KEY] = {}

len(cache.keys())

2

In [7]:
questions = list(mediqa_df['question'].tolist())
expected_answers = list(mediqa_df['answer'].tolist())
predicted_answers = []

for index, question in tqdm(enumerate(questions)):
  if question not in cache[CACHE_KEY]:
    cache[CACHE_KEY][question] = app.invoke({'question': question})['generation']

  predicted_answers.append(cache[CACHE_KEY][question])

  with open(cache_path, 'w') as f:
    json.dump(cache, f)

cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
bleu_score = bleu_metric(expected_answers, predicted_answers)
rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
factscore_score = factscore_metric(expected_answers, predicted_answers)
summac_zs_score = summac_zs_metric(expected_answers, predicted_answers)
summac_conv_score = summac_conv_metric(expected_answers, predicted_answers)

cos_score, bleu_score, rogue_1_score, rogue_l_score, factscore_score, summac_zs_score, summac_conv_score

0it [00:00, ?it/s]

2026-01-14 01:00:34,174 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 01:00:37,022 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 01:00:38,851 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 01:00:49,585 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 01:00:55,935 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 01:01:10,228 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 01:01:53,607 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 01:02:13,431 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 01:02:21,835 - INFO - HTTP Request: POST https://openrouter.a

Error fetching medrxiv papers: HTTPSConnectionPool(host='api.biorxiv.org', port=443): Read timed out. (read timeout=30)


2026-01-14 02:35:42,990 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 02:36:03,183 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 02:36:24,224 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 02:36:29,356 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 02:36:31,429 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 02:37:17,746 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 02:37:25,417 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 02:37:27,893 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-14 02:37:48,393 - INFO - HTTP Request: POST https://openrouter.a

Extracting facts from generations...


...

Generating decisions...


0it [00:00, ?it/s]

Error computing FActScore for pair: division by zero


<All keys matched successfully>


(0.7518032486300212,
 0.020094029678274238,
 0.2635365825979728,
 0.13205694570405652,
 0.0,
 -0.30403565838528157,
 0.24256826231354162)